# 03 — Manifest → priors → scale-conditioned mass regressor (GPU job 2)

The novel model: mass regression conditioned on measured metric geometry (docs/MODELS.md §4). Self-contained — stages its own copy of Nutrition5k.

First it **stages Nutrition5k to local disk**, then runs the three pipeline steps: (1) extract the geometry manifest from the RGB-D captures — the same quantities the phone measures; (2) fit the κ/φ/h̄ shape priors used by the pure-geometry pipeline; (3) train the regressor.

- **Storage — read this:** the raw dataset lives on **local disk**, not Drive. Both the manifest extraction and *every training epoch* read the ~5k per-dish RGB-D folders, and that many-small-files workload aborts over Drive's FUSE mount (`Errno 103`, `ECONNABORTED`). Drive is only for the handful of big files worth keeping — the **outputs** (manifest, priors, checkpoint). The manifest bakes in the local `/content/n5k/...` image paths, so a fresh session must rerun the staging cell before re-training.
- **Runtime:** dataset staging minutes–~1 h (bandwidth); manifest extraction ~30–60 min (CPU-bound); training ~1–2 h on H100 at batch 128.
- **Benchmarks to beat:** 26.1% calorie MAPE (RGB) / 16.5% (RGB+depth).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/physics-powered-portion-estimation'

# The raw dataset AND framework caches stay on LOCAL disk. Drive's FUSE mount
# aborts (Errno 103, ECONNABORTED) on the per-dish RGB-D workload — thousands
# of tiny files listed and read by both the manifest extraction and every
# training epoch — and mmap over FUSE is unreliable too. Drive is only for the
# few big files worth keeping: the OUTPUTS below. The next cell stages the
# dataset into DATA from the public bucket (reliable HTTPS, resumable).
os.environ['HF_HOME'] = '/content/hf-cache'
os.environ['TORCH_HOME'] = '/content/torch-cache'

DATA = '/content/n5k'                     # local SSD — NOT Drive (see above)
OUT = f'{DRIVE_ROOT}/out'                 # outputs persist to Drive
CKPT = f'{DRIVE_ROOT}/checkpoints/mass-regressor.pt'
!mkdir -p "{OUT}" "{DATA}"
!nvidia-smi | head -12

In [2]:
# Clone the repo. Private repo → add your GitHub token as a Colab secret named
# GH_TOKEN (🔑 left sidebar). Public repo → no token needed. Fails LOUD.
import os, subprocess
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN') or ''
except Exception:
    pass
repo = 'github.com/Hakeem-Hannoon/physics-powered-portion-estimation.git'
url = f'https://x-access-token:{token}@{repo}' if token else f'https://{repo}'
# A prior run may leave the kernel's CWD inside /content/ppe (the %cd in the
# next cells). Deleting that dir out from under the kernel makes the git clone
# below fail with "Unable to read current working directory". Reset to a
# stable dir first so re-runs are safe.
os.chdir('/content')
subprocess.run(['rm', '-rf', '/content/ppe'])
r = subprocess.run(['git', 'clone', '--depth', '1', url, '/content/ppe'],
                   capture_output=True, text=True)
assert os.path.isdir('/content/ppe/model'), (
    (r.stderr.replace(token, '***') if token else r.stderr) +
    '\nClone failed — add a GH_TOKEN Colab secret (🔑) or make the repo public.')
print('repo ready')

%pip -q install timm pandas

repo ready
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 74.0 MB/s eta 0:00:00


In [ ]:
# Stage Nutrition5k (imagery + metadata + splits) to LOCAL disk straight from
# the public bucket over HTTPS — NOT from Drive. Reading the ~5k per-dish RGB-D
# folders off Drive's FUSE mount aborts (Errno 103); local disk is reliable and
# fast, and it's what every training epoch reads. Resumable: reruns skip files
# already present. ~15–25 GB. Same downloader as notebook 01, pointed at DATA.
import concurrent.futures, pathlib, requests

BUCKET = 'nutrition5k_dataset'
# Only the slice the pipeline reads — overhead RGB-D, mass/kcal metadata, split
# ids — not the ~160 GB of side-angle video.
PREFIXES = [
    'nutrition5k_dataset/imagery/realsense_overhead/',
    'nutrition5k_dataset/metadata/',
    'nutrition5k_dataset/dish_ids/',
]
API = f'https://storage.googleapis.com/storage/v1/b/{BUCKET}/o'


def list_objects(prefix):
    """Page through the bucket's JSON listing API, yielding (name, size) for
    every object under `prefix`. Unauthenticated GET — the bucket is public."""
    token = None
    while True:
        # maxResults caps the page size; nextPageToken walks to the next page.
        params = {'prefix': prefix, 'fields': 'items(name,size),nextPageToken', 'maxResults': 5000}
        if token:
            params['pageToken'] = token
        r = requests.get(API, params=params, timeout=60)
        r.raise_for_status()
        payload = r.json()
        for item in payload.get('items', []):
            yield item['name'], int(item.get('size', 0))
        token = payload.get('nextPageToken')
        if not token:   # no more pages — done
            break


def fetch(job):
    """Download one object unless an identical-size copy already exists — that
    size check IS the resumability, so a rerun skips completed files. Streams to
    a .part temp and atomically renames, so an interrupted transfer never leaves
    a truncated file a later run would mistake for complete."""
    name, size = job
    dest = pathlib.Path(DATA) / name.removeprefix('nutrition5k_dataset/')
    if dest.exists() and dest.stat().st_size == size:
        return 0  # already staged — skip
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f'https://storage.googleapis.com/{BUCKET}/{name}'
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + '.part')
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(1 << 20):   # 1 MiB chunks
                f.write(chunk)
        tmp.rename(dest)   # atomic swap — the file appears only once fully written
    return 1


# Enumerate everything up front (to print a total), then fetch in parallel —
# 8 threads saturates the bucket without much memory.
jobs = [j for p in PREFIXES for j in list_objects(p)]
print(f'{len(jobs)} objects, {sum(s for _, s in jobs) / 1e9:.1f} GB → {DATA}')
staged = 0
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
    for i, n in enumerate(pool.map(fetch, jobs), 1):
        staged += n
        if i % 500 == 0:
            print(f'{i}/{len(jobs)} checked, {staged} downloaded')
print(f'staged to local disk: {staged} new files ({len(jobs) - staged} already present)')

In [ ]:
# Step 1 — geometry manifest. Runs model/data/prepare_nutrition5k.py: for every
# dish it fits the table plane from the depth map, measures area/height/volume
# (the same physics the phone computes at capture time), and joins the mass/kcal
# labels into one CSV. Reads from local DATA; writes the CSV to OUT on Drive.
# ~30–60 min, CPU-bound. `head -3` is a sanity peek at the columns.
%cd /content/ppe
!python model/data/prepare_nutrition5k.py --root "{DATA}" --out "{OUT}/n5k-manifest.csv"
!head -3 "{OUT}/n5k-manifest.csv"

In [ ]:
# Step 2 — shape priors. Runs model/priors/fit_priors.py over the manifest to
# fit the per-class constants κ (V = κ·A^1.5), φ (prism fill fraction), and h̄
# (typical height). The printed global κ replaces DEFAULT_KAPPA in
# packages/pipeline/src/estimate.ts and feeds nutrition/'s shape_priors table.
# Seconds to run; writes priors.json to OUT on Drive.
!python model/priors/fit_priors.py --manifest "{OUT}/n5k-manifest.csv" --out "{OUT}/priors.json"
!cat "{OUT}/priors.json"

In [ ]:
# Step 3 — train the scale-conditioned mass regressor. Runs
# model/train/mass_regressor_nutrition5k.py: a CNN backbone reads each RGB crop,
# FiLM modulates its features by the measured physics (area/height/scale), and a
# head regresses log(mass) + log(kcal). Per-epoch mass/kcal MAPE prints; the
# best-by-mass-MAPE checkpoint saves to CKPT on Drive. ~1–2 h on an H100.
!python model/train/mass_regressor_nutrition5k.py \
  --manifest "{OUT}/n5k-manifest.csv" \
  --epochs 50 --batch-size 128 \
  --output "{CKPT}"
# Paste the printed best MAPE into the README 'Testing set & results' table:
print('README row template:')
print('| Nutrition5k test split | calorie MAPE | 26.1% RGB / 16.5% depth | **<best kcal MAPE from above>** |')